# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# Unit of analysis: ONE ROW = one content page (content_hash_id), for one client
# (client_hash_id), measured over a defined trailing window.
#
# Table: fact_content_daily_performance (joined with dim_content for metadata)
#
# Time window: I will use a mid-panel month, month=2026-03, for feature
# development (as instructed - the freshest month is reserved as a sealed
# test window and must not be used to build label logic).
#
# This matches my Lane 2 question: which pages should a review team look at
# first for refresh/expansion/protection - the grain (one page) is exactly
# what a reviewer would open and act on.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [2]:
# The Data Contract (5 answers):
#
# 1. What one row means: one content page, for one client, on one day -
#    aggregated over my chosen feature window (prior 90 days).
#
# 2. Table(s) used: fact_content_daily_performance (daily facts) joined to
#    dim_content (content metadata) on content_hash_id.
#
# 3. Time window: features built from month=2026-03 (mid-panel, safe for
#    label development). The final month (2026-06) is reserved as a sealed
#    test window later - never used to shape label logic now.
#
# 4. Label/proxy: a page is a "review candidate" if it shows declining
#    trend/visibility with meaningful demand (impressions) in the feature
#    window - this is a proxy, not a guaranteed cause of future recovery.
#
# 5. Deliberately excluded: I exclude any FlyRank product decision fields
#    (health_score, priority_score, action_type) - these are not in the
#    warehouse release anyway, but I confirm I will never reconstruct and
#    feed them back in as a feature, since that would just teach the model
#    to copy an existing rule instead of finding real signal.x

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import os
from google.colab import userdata
import duckdb

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

# Query 1: grain check - is one row really "one page, one client, one day"?
q1 = con.sql("""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain check (empty = grain confirmed correct):")
print(q1)

# Query 2: row count and date span for the slice
q2 = con.sql("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print("\nRow count and date span:")
print(q2)

# Query 3: availability - filter with IS TRUE
q3 = con.sql("""
    SELECT COUNT(*) as rows_with_ga4
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()
print("\nRows with GA4 data available:")
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain check (empty = grain confirmed correct):
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, row_count]
Index: []

Row count and date span:
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Rows with GA4 data available:
   rows_with_ga4
0         413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [4]:
# Data limits (named honestly):
#
# 1. Unbalanced panel: different clients have different amounts of history -
#    some clients have 12+ months, others much less. A page's "trend" looks
#    different depending on how much history exists for its client.
#
# 2. GSC-only early rows: before a client's GA4 tracking started, rows only
#    have search (GSC) data - ga4_data_available = FALSE. Treating these as
#    "zero traffic" instead of "not tracked yet" would be a mistake.
#
# 3. Window overlap risk: if my feature window and target window ever share
#    days, that's leakage - I must keep them strictly separated (e.g., prior
#    90 days for features, a separate later window for any future label).
#
# 4. This slice cannot prove causation: even a strong pattern only tells me
#    what is associated with review-worthy pages, not that fixing them will
#    cause recovery - that would need a controlled experiment.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.